# [1.6] Local Frontier ML Infrastructure - Exercises

**Core Question.** How do you keep frontier interpretability notebooks honest on one local GPU?

This notebook builds the contract later extension sections depend on: environment metadata, memory estimates, parity checks, deterministic generation checks, disk-backed activation storage, and a real CUDA report.

## Learning Objectives

- Print reproducible Python, PyTorch, CUDA, and GPU metadata.
- Separate memory estimates from measured peak CUDA allocation.
- Detect both logit parity matches and real logit drift.
- Save activations with metadata and reload them from disk.
- Read the committed CUDA report without turning it into a model-quality claim.

> ```yaml
> Difficulty: 3
> Importance: 5
> ```

<details>
<summary>Help - why this matters</summary>

A later interpretability result is only credible if the notebook can show which environment ran it, which checkpoint or tensor path executed, how much memory it used, and which controls failed.

</details>

In [ ]:
GT_TIER = "GT-1"
EXERCISE_ID = "1_6_local_frontier_ml_infrastructure"
DIFFICULTY = 3
IMPORTANCE = 5
EXPECTED_RUNTIME = "seconds for toy contracts; minutes for the local CUDA runtime report"
REQUIRES_GPU = True

import json
import sys
import tempfile
from pathlib import Path

import torch as t

chapter = "chapter1_transformer_interp"
section = "part6_frontier_ml_infrastructure"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part6_frontier_ml_infrastructure.tests as tests
import part6_frontier_ml_infrastructure.utils as utils

from arena_ext import (
    DiskActivationStore,
    compare_logits,
    deterministic_generation_equal,
    estimate_inference_memory,
    get_environment_report,
)

MAIN = True

## Exercise 1 - Environment Checks

Print the local runtime report and warnings for a required VRAM estimate.

<details>
<summary>Expected output</summary>

```text
cuda_available: True
cuda_version: 13.2
torch: 2.12.1+cu132
bf16_supported: True
```

</details>

<details>
<summary>Help - what not to hide</summary>

Do not silently accept a GPU-required notebook when CUDA is missing. CPU smoke mode is useful, but it is not the acceptance path.

</details>

<details>
<summary>Solution</summary>

Call `get_environment_report()`, print `report.as_dict()`, print warnings, and return the report object.

</details>

In [ ]:
def run_environment_check(required_vram_gb: float | None = 24.0):
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


report = run_environment_check(required_vram_gb=24.0)
report.as_dict()

## Exercise 2 - VRAM Budget Estimates

Estimate a 1B-parameter BF16 smoke-test model with KV-cache and activation memory.

<details>
<summary>Expected output</summary>

```text
total_gb: about 3.66
Fits 24GB local tier: True
All tests in `test_memory_budget_fits_local_tier` passed!
```

</details>

<details>
<summary>Help - estimate versus measurement</summary>

A budget estimate catches obvious local-tier mistakes before model load. It is not a replacement for `torch.cuda.max_memory_allocated()` after a real run.

</details>

<details>
<summary>Solution</summary>

Use `estimate_inference_memory` with 1B BF16 parameters, batch size 1, context 2048, hidden size 2048, 18 layers, 8 KV heads, head dim 256, and 1.5GB overhead.

</details>

In [ ]:
def estimate_gemma_1b_smoke_budget(context_length: int = 2048):
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


tests.test_memory_budget_fits_local_tier(estimate_gemma_1b_smoke_budget)
tests.test_memory_budget_rejects_oversized_model()

## Exercise 3 - HF Parity and Drift Controls

Compute logit-level parity metrics and wrap them in a passing smoke test.

<details>
<summary>Expected output</summary>

```text
All tests in `test_compare_logits_detects_match` passed!
All tests in `test_compare_logits_rejects_real_drift` passed!
```

</details>

<details>
<summary>Help - why argmax is not enough</summary>

Two implementations can share an argmax while having a distribution drift large enough to break interventions. Track max error, MSE, KL, and top-k agreement.

</details>

<details>
<summary>Solution</summary>

Generate fixed reference logits, add a tiny perturbation, run `compare_logits`, and require explicit tolerances for max error, MSE, KL, and top-k agreement.

</details>

In [ ]:
tests.test_compare_logits_detects_match(compare_logits)
tests.test_compare_logits_rejects_shape_mismatch(compare_logits)
tests.test_compare_logits_rejects_real_drift(compare_logits)


def hf_parity_smoke_test() -> bool:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


tests.test_hf_parity_smoke_test_passes(hf_parity_smoke_test)

## Exercise 4 - Generation Parity

Use exact token equality for deterministic greedy-generation checks.

<details>
<summary>Expected output</summary>

```text
All tests in `test_deterministic_generation_equal_detects_mismatch` passed!
```

</details>

<details>
<summary>Help - where exact equality applies</summary>

Exact token equality is appropriate for fixed greedy decoding. Do not expect it from temperature sampling.

</details>

<details>
<summary>Solution</summary>

Return `deterministic_generation_equal([1, 2, 3, 4], t.tensor([1, 2, 3, 4]))`.

</details>

In [ ]:
tests.test_deterministic_generation_equal_detects_mismatch(
    deterministic_generation_equal,
)


def generation_parity_smoke_test() -> bool:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE

## Exercise 5 - Activation Storage

Write and reload activation shards with metadata instead of keeping every activation in VRAM.

<details>
<summary>Expected output</summary>

```text
num_records: 2
names: ['resid_pre', 'mlp_out']
All tests in `test_activation_store_smoke_test_contract` passed!
```

</details>

<details>
<summary>Help - what metadata prevents</summary>

A tensor file without hook, layer, prompt-set, and run metadata is hard to audit later. The index is part of the evidence.

</details>

<details>
<summary>Solution</summary>

Create a `DiskActivationStore`, append two tensors with hook/layer metadata, reload both tensors, and return `store.summary()`.

</details>

In [ ]:
def activation_store_smoke_test(output_dir: str | Path = "activation_store_smoke") -> dict:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


with tempfile.TemporaryDirectory() as tmpdir:
    tests.test_disk_activation_store_roundtrip(Path(tmpdir))
    tests.test_activation_store_smoke_test_contract(
        Path(tmpdir),
        activation_store_smoke_test,
    )

## Exercise 6 - Notebook Contract

Return a JSON-serializable smoke-test dictionary with environment, budget, parity, generation, and activation-store evidence.

<details>
<summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

</details>

<details>
<summary>Solution</summary>

Build the budget, run the parity helpers, write activations in a temporary directory, and return all results in a dictionary.

</details>

In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


tests.test_notebook_contract(run_smoke_test)

## Signature Result

The committed CUDA report is the section's signature result: the managed uv environment runs a real BF16 CUDA tensor path and records its environment and peak VRAM.

| Check | Observed | Required |
|---|---:|---:|
| Python / PyTorch | `3.14.6` / `2.12.1+cu132` | `3.14` / `2.12.1+cu132` |
| CUDA runtime | `13.2` | `13.2` |
| BF16 CUDA matmul | `[1024,1024]`, finite | finite BF16 tensor |
| `uv pip check` | pass | pass |
| Peak VRAM | `0.043 GB` | `< 1 GB` |

<details>
<summary>Help - what this proves</summary>

It proves local runtime readiness, not correctness of a future model-specific implementation.

</details>

## Limitations

This notebook does not load a real HF checkpoint, benchmark a model, or prove any later interpretability result. Later sections need their own toy oracles, reference parity, and negative controls.

## Bonus - Anomaly Hunting

- Increase the context length and watch the estimate change before loading anything.
- Change one top logit and confirm parity fails even if shapes match.
- Remove activation metadata and decide what evidence becomes unauditable.

In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    gpu = report["metrics"]["gpu_test"]
    assert report["accepted"]
    assert gpu["cuda_available"]
    assert gpu["gpu_tensor_test_passed"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


tests.test_committed_gpu_report_records_cuda_runtime_and_no_fallback()
tests.test_exercise_notebook_course_ready_surface()